# Sesión 6 · AC_PR_6_1 · Calidad del aire y pronóstico ARIMA
**Sistemas de adquisición de datos · TSU Electrónica y Pruebas · Jabil · 2027**

Datos públicos reales del SIMAJ (Sistema de Monitoreo Atmosférico de Jalisco), 2022, una medición por hora. No necesitas programar: ejecuta cada celda con ▶ en orden y cambia solo los campos del formulario.

In [ ]:
# @title Preparación (ejecuta esta celda primero; no hay nada que cambiar)
import io, re, warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
warnings.filterwarnings("ignore")
MESES = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]
def dia(t): return f"{t.day:02d} {MESES[t.month - 1]}"
plt.rcParams.update({"figure.figsize": (12, 4.2), "font.size": 12,
                     "axes.spines.top": False, "axes.spines.right": False})
URL = "https://github.com/ggt13/JABIL_DATA_24A/raw/main/sesion_06_series_de_tiempo_y_pronostico/datos/datos_2022.xlsx"
HOJA = "SIMAJ2022"
VARIABLE = "PM10"
LIMITE = 50.0          # µg/m³, promedio de 24 h (NOM-025-SSA1-2021, desde 2026)
HORAS_MIN = 18          # 75 % de 24 h
DIAS_ENTRENA, DIAS_PRUEBA = 60, 14
EQUIPOS = {"1": ["Aguilas", "2022-03-30"], "2": ["Centro", "2022-03-15"], "3": ["Las Pintas", "2022-03-02"], "4": ["Miravalle", "2022-04-26"], "5": ["Santa Fe", "2022-03-02"], "6": ["Tlaquepaque", "2022-04-05"], "7": ["Aguilas", "2022-09-20"], "8": ["Santa Fe", "2022-11-15"], "9": ["Centro", "2022-05-10"], "10": ["Las Pintas", "2022-11-29"]}
print("Listo. Sigue con el bloque A.")


## Bloque A · El archivo tal como llega
Mira cuántas celdas no son números y qué códigos aparecen. ¿Qué estaciones no sirven para PM10?

In [ ]:
# @title Bloque A · El archivo tal como llega
fuente = "GitHub (archivo del curso)"  # @param ["GitHub (archivo del curso)", "subir mi archivo"]
if "crudo" not in globals() or globals().get("_fuente") != fuente:   # se descarga una sola vez (≈ 30 s)
    if fuente == "subir mi archivo":
        from google.colab import files
        nombre, contenido = list(files.upload().items())[0]
        crudo = pd.read_excel(io.BytesIO(contenido), sheet_name=HOJA)
    else:
        print("Descargando el archivo del SIMAJ (≈ 7 MB)…")
        crudo = pd.read_excel(URL, sheet_name=HOJA)
    _fuente = fuente
medidas = [c for c in crudo.columns if c not in ("Estacion", "Date_Time", "Hora")]
num = crudo[medidas].apply(pd.to_numeric, errors="coerce")
print(f"{len(crudo):,} renglones · {crudo['Estacion'].nunique()} estaciones · {len(medidas)} variables")
vacias = crudo[medidas].isna()
print(f"Celdas de medición: {100 * num.notna().values.mean():.0f} % con número, "
      f"{100 * (~vacias & num.isna()).values.mean():.0f} % con código, {100 * vacias.values.mean():.0f} % vacías")
pila = crudo[medidas].stack()
codigos = pila[pd.to_numeric(pila, errors="coerce").isna()].astype(str)
print("\nCódigos que aparecen en lugar de números (cuántas veces):")
print(codigos.value_counts().to_string())
display(crudo.head(8))
disp = crudo.assign(ok=pd.to_numeric(crudo[VARIABLE], errors="coerce").notna()).groupby("Estacion")["ok"].mean() * 100
ax = disp.sort_values().plot.barh(color=["#C62828" if v < 40 else "#D9822B" if v < 75 else "#2E7D32" for v in disp.sort_values()])
ax.set_xlabel(f"% de horas con {VARIABLE} válido en el año"); ax.axvline(75, color="#444", ls="--"); plt.show()


## Bloque B · Limpieza de tu estación
Elige tu número de equipo. El bloque arma la fecha y hora, borra duplicados, convierte los códigos en vacío e interpola solo huecos cortos. El promedio diario vale si hay al menos 18 horas.

In [ ]:
# @title Bloque B · Limpieza y promedio diario de tu estación
equipo = 1  # @param {type:"slider", min:1, max:10, step:1}
estacion, inicio = EQUIPOS[str(equipo)]
g = crudo[crudo["Estacion"] == estacion].copy()
print(f"Equipo {equipo}: estación {estacion}\n")
# 1. Fecha y hora: Date_Time + Hora (en algunas estaciones Date_Time trae solo la fecha)
rep_antes = int(g.duplicated("Date_Time").sum())
g["t"] = pd.to_datetime(g["Date_Time"].dt.strftime("%Y-%m-%d") + " " + g["Hora"].astype(str))
rep_t = int(g.duplicated("t").sum())
print(f"1. Renglones con Date_Time repetido: {rep_antes}.  Con fecha + hora: {rep_t}.")
# 2. Duplicados reales fuera
g = g.drop_duplicates("t").set_index("t").sort_index()
print(f"2. Duplicados borrados: {rep_t}.  Quedan {len(g)} horas.")
# 3. Códigos → vacío
cods = g[VARIABLE][pd.to_numeric(g[VARIABLE], errors="coerce").isna()].astype(str).value_counts()
x = pd.to_numeric(g[VARIABLE], errors="coerce")
print(f"3. Horas con código en lugar de número: {int(cods.sum())}  ({', '.join(f'{k}: {v}' for k, v in cods.items())})")
print(f"   Horas válidas: {x.notna().mean() * 100:.1f} %")
# 4. Huecos de hasta 3 horas: interpolación lineal (solo para ver la serie horaria; no cuentan para el 75 %)
hueco = x.isna()
largo = hueco.groupby((hueco != hueco.shift()).cumsum()).transform("sum")
xi = x.interpolate(limit_area="inside").where(~hueco | (largo <= 3))
print(f"4. Horas interpoladas (huecos ≤ 3 h): {int(xi.notna().sum() - x.notna().sum())}")
# 5. Promedio diario con el criterio del 75 %
n = x.resample("D").count(); diario = x.resample("D").mean().where(n >= HORAS_MIN)
print(f"5. Días con promedio válido (≥ {HORAS_MIN} h): {int(diario.notna().sum())} de {len(diario)}")
print(f"   Días por encima de {LIMITE:.0f} µg/m³: {int((diario > LIMITE).sum())}")
fig, (a1, a2) = plt.subplots(2, 1, figsize=(12, 7))
mes = pd.Timestamp(inicio) - pd.Timedelta(days=30)
a1.plot(xi[mes: mes + pd.Timedelta(days=10)], color="#9DB7D5", label="interpolado")
a1.plot(x[mes: mes + pd.Timedelta(days=10)], color="#1F4E79", label="medido")
a1.set_ylabel(f"{VARIABLE} horario"); a1.legend(frameon=False)
a2.plot(diario, color="#D9822B"); a2.axhline(LIMITE, color="#C62828", ls="--")
a2.set_ylabel(f"{VARIABLE} diario [µg/m³]")
plt.tight_layout(); plt.show()


## Bloque C · Pronóstico
Se entrena con 60 días y se pronostica un día adelante durante 14 días. Empieza con ARIMA(1, 0, 0). Después prueba otros órdenes y anota su AIC: el orden se elige con el AIC del entrenamiento (menor es mejor), no viendo los días de prueba.

In [ ]:
# @title Bloque C · Pronóstico un día adelante: ARIMA contra persistencia
p = 1  # @param [0, 1, 2, 3] {type:"raw"}
d = 0  # @param [0, 1] {type:"raw"}
q = 0  # @param [0, 1, 2] {type:"raw"}
p, d, q = int(p), int(d), int(q)
ini = pd.Timestamp(inicio)
entrena = diario[ini - pd.Timedelta(days=DIAS_ENTRENA): ini - pd.Timedelta(days=1)]
prueba = diario[ini: ini + pd.Timedelta(days=DIAS_PRUEBA - 1)]
print(f"Entrenamiento: {dia(entrena.index[0])} a {dia(entrena.index[-1])} ({int(entrena.isna().sum())} días sin dato válido: ARIMA los tolera)")
print(f"Prueba: {dia(prueba.index[0])} a {dia(prueba.index[-1])}\n")
modelo = ARIMA(entrena.values, order=(p, d, q)).fit()
print(f"AIC del entrenamiento: {modelo.aic:.1f}  (para elegir el orden se compara el AIC: menor es mejor)\n")
todo = np.concatenate([entrena.values, prueba.values])
pron = modelo.apply(todo).predict(start=len(entrena), end=len(todo) - 1)   # cada día usa solo los anteriores
pers = np.concatenate([[entrena.values[-1]], prueba.values[:-1]])
real = prueba.values
def mae(e): return float(np.mean(np.abs(e)))
def rmse(e): return float(np.sqrt(np.mean(e ** 2)))
res = pd.DataFrame({"Día": [dia(t) for t in prueba.index], "Real": real.round(1), "Persistencia": pers.round(1),
                    f"ARIMA({p},{d},{q})": pron.round(1)})
display(res)
ac_m = int(((pron > LIMITE) == (real > LIMITE)).sum()); ac_p = int(((pers > LIMITE) == (real > LIMITE)).sum())
print(f"ARIMA({p},{d},{q}):  MAE = {mae(real - pron):.2f}   RMSE = {rmse(real - pron):.2f} µg/m³   aciertos arriba/abajo de {LIMITE:.0f}: {ac_m} de {DIAS_PRUEBA}")
print(f"Persistencia:   MAE = {mae(real - pers):.2f}   RMSE = {rmse(real - pers):.2f} µg/m³   aciertos: {ac_p} de {DIAS_PRUEBA}")
cola = entrena[-21:]
plt.plot(cola.index, cola.values, "o-", color="#9DB7D5", ms=4, label="entrenamiento")
plt.plot(prueba.index, real, "o-", color="#1F4E79", lw=2, label="real")
plt.plot(prueba.index, pron, "s--", color="#D9822B", lw=2, label=f"ARIMA({p},{d},{q})")
plt.plot(prueba.index, pers, "^:", color="#777", label="persistencia")
plt.axhline(LIMITE, color="#C62828", ls="--"); plt.ylabel(f"{VARIABLE} [µg/m³]"); plt.legend(frameon=False, ncol=4); plt.show()


## Bloque D · Para tu hoja
Con ARIMA(1, 0, 0), pega estos seis valores en la pestaña «Práctica»: deben marcar ✔.

In [ ]:
# @title Bloque D · Valores para tu hoja (pestaña «Práctica», columna E)
if (p, d, q) != (1, 0, 0):
    print("Para comparar con tu hoja vuelve a ejecutar el bloque C con p = 1, d = 0, q = 0.")
else:
    mu, phi = modelo.params[0], modelo.params[1]
    print(f"Equipo {equipo} · {estacion}\n")
    for nom, v in (("μ", round(mu, 3)), ("φ", round(phi, 4)), ("MAE AR(1)", round(mae(real - pron), 2)),
                   ("RMSE AR(1)", round(rmse(real - pron), 2)), ("MAE persistencia", round(mae(real - pers), 2)),
                   ("RMSE persistencia", round(rmse(real - pers), 2))):
        print(f"{nom:<18} {v}")
